# Group 2

# Project: AI-Powered Job Scam Detection System

## Group Members

- Briannah Chelangat
- Christopher Karuiki
- Melisa Achieng
- Cleopas Karanja
- Alex Kinyua

---

## 1. Business Understanding

### 1.1 Business Problem

Kenya has experienced a significant rise in fraudulent job advertisements, particularly targeting unemployed youth seeking local and overseas employment opportunities. Fraudsters use fake job postings to:

- Collect application fees
- Steal personal information
- Conduct financial scams
- Facilitate human trafficking through fake overseas jobs

Current interventions are mainly reactive, where agencies are investigated or blacklisted after victims have already suffered losses. Job seekers lack a reliable tool that can assess the legitimacy of a job advertisement before they apply or pay money.

### 1.2 Business Objectives

The project seeks to:

#### Objective 1: Detect Fraudulent Job Advertisements

Develop a machine learning model that classifies job advertisements as: Genuine (0) or Fraudulent (1) using textual and structured job-posting information.

#### Objective 2: Identify Scam Patterns

Use NLP techniques to identify common scam indicators such as: Urgent hiring language, Upfront payment requests, Poorly written descriptions, Unrealistic salaries and Suspicious contact information.

#### Objective 3: Integrate Kenyan Government Verification

The system should verify:

- Whether a recruitment agency is licensed
- Whether the agency appears on government blacklists
- Whether the company exists in government records

#### Objective 4: Explain Predictions

Instead of only predicting fraud, the system should explain why a posting was flagged with reasons like ; Agency not found in NEA registry, Requests visa processing fee, Unrealistic overseas salary, Uses urgency phrases such as "Apply Immediately" e.t.c

### 1.3 Stakeholders

**Primary stakeholders**

1. **Job Seekers**  
   Use the system before applying for jobs. Helps them to avoid scams, protect personal data and avoid financial losses.
2. **Government Agencies**  
   Government agencies like National Employment Authority (NEA), Ministry of Labour and DCI will use the system to detect scam patterns earlier and monitor suspicious agencies.
3. **Recruitment Platforms**  
   Recruitment platforms like BrighterMonday, Fuzu and MyJobsInKenya will use the platform to automatically flag suspicious suspicious job posts before publication.

### 1.4 Success Criteria

The project will be considered successful if it achieves:

- **High Recall:** The model should identify most scam jobs.  
  Recall is important because missing a scam may result in real-world harm to users.
- **Good Precision:** Legitimate jobs should not be incorrectly flagged too often.
- **High F1 Score:** Balances precision and recall.
- **Strong ROC-AUC:** To measure overall classification performance.
- **Explainability:** Predictions should be understandable to non-technical users.

---

## 2. Data Understanding

### 2.1 Data Sources

- **Primary Dataset:** DIFrauD / EMSCAD
- **Dataset used for training had the following characteristics:**
  - 14,295 job postings
  - 599 fraudulent postings
  - 13,696 legitimate postings
- **Label:**
  - `1` = Fraudulent
  - `0` = Legitimate
- **Kenya Specific Verification Data:**
  - **NEA Recruitment Agency Register:** Used to determine whether an agency is licensed.
  - **DCI Blacklisted Agencies:** Used to identify agencies involved in fraud.
  - **Government Advisories:** Used to identify emerging scam patterns.
  - **Local Job Boards** like BrighterMonday, Fuzu and MyJobsInKenya: Used for future model adaptation.

### 2.2 Dataset Variables

The project standardizes job advertisements into a common structure such as:

| Variable | Description |
| --- | --- |
| `posting_id` | Unique job identifier |
| `title` | Job title |
| `description` | Job description |
| `company` | Hiring company |
| `agency` | Recruitment agency |
| `location` | Job location |
| `salary` | Salary offered |
| `employment_type` | Full-time, part-time, contract |
| `email` | Contact email |
| `source` | Source platform |
| `country` | Job destination country |
| `is_overseas` | Overseas job indicator |
| `fraud_label` | Target variable |

### 2.3 Exploratory Data Analysis (EDA)

**Key questions:**

- **Class Distribution:** How many jobs are fraud versus genuine?
- **Missing Values:** Which variables contain missing data?
- **Salary Analysis:** Do scam jobs offer unusually high salaries?
- **Text Analysis:** Which words frequently appear in scam advertisements? Examples: urgent, visa, fee, immediate and no experience.
- **Country Analysis:** Which overseas destinations are most commonly associated with suspicious advertisements?

---

## 3. Data Preparation

### 3.1 Data Cleaning

- **Remove Duplicates:** Many scam ads are reposted multiple times.
- **Handle Missing Values** like Missing salary, Missing company name and Missing agency information
- **Standardize Text by converting:** Uppercase to lowercase, Remove punctuation, Remove HTML tags and Remove special characters.

### 3.2 Feature Engineering

#### A. Text Features (NLP)

Transform text into machine-readable features.

**Methods:**

- **TF-IDF:** Measures the importance of words in a job advertisement.
- **Word Embeddings:** Capture semantic meaning.
- **Sentence Embeddings:** Used in transformer-based models.

#### B. Rule-Based Features

**Examples:**

| Rule | Flag |
| --- | --- |
| Contains "pay registration fee" | 1 |
| Contains "visa processing fee" | 1 |
| Uses Gmail instead of company domain | 1 |
| Unrealistic salary | 1 |
| Missing company information | 1 |

#### C. Government Verification Features

**Examples:**

| Feature | Meaning |
| --- | --- |
| `agency_verified` | Agency exists in NEA registry |
| `agency_blacklisted` | Agency appears on blacklist |
| `company_verified` | Company exists |

### 3.3 Dataset Integration (2-Class, Merged)

This notebook standardizes the available job-posting data, harmonizes labels, and exports one modelling-ready dataset.

**Target scheme:** `is_fraud` has two classes:

- **0: Legitimate**
- **1: Scam** (postings labelled Suspicious or Fraudulent in the source files are both Scam)

> **Important:** real job-board postings are not automatically labeled legitimate. Their `is_fraud` remains `NaN` unless the project team approves that policy. EMSCAD provides real labels, while synthetic data provides generated training scenarios.

## Data-preparation pipeline (execution order)

This notebook is written to run top to bottom from a fresh kernel. Every step only uses objects created by the steps above it, and `final_df` is created **once** (Part 12) and then only cleaned.

| Part | Step |
| --- | --- |
| 1 | Project setup: imports, seeds, configuration, helper functions |
| 2 | Load all source datasets |
| 3 | Initial dataset inspection |
| 4 | Standardize column names |
| 5 | Clean string / text fields |
| 6 | Handle missing values |
| 7 | Normalize categorical values |
| 8 | Convert data types and validate labels |
| 9 | Parse structured fields (salary, email) |
| 10 | Source-specific cleaning of the two synthetic files |
| 11 | Standardize every source into the shared schema |
| 12 | Combine datasets and check labels |
| 13 | Exact duplicate removal |
| 14 | Near-duplicate (fingerprint) removal |
| 15 | Final data-quality checks |
| 16 | Leakage and feature-preparation note |
| 17 | EDA-ready dataset (`eda_df`) |
| 18 | Export |
| 19 | Final automated check |

## Part 1: Project setup

Import libraries

In [41]:
import hashlib
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [42]:
DATA_DIR = Path(".")

SOURCE_FILES = {
    "Fake Job Postings":        DATA_DIR / "fake_job_postings.csv",
    "Fuzu Kenya":               DATA_DIR / "fuzu_kenya_jobs.csv",
    "PigiaMe Kenya":            DATA_DIR / "pigiame_jobs_full.csv",
    "Corporate Staffing Kenya": DATA_DIR / "corporatestaffing_jobs_full.csv",
    "BrighterMonday Kenya":     DATA_DIR / "brightermonday_kenya_jobs.csv",
    "JobWeb Kenya":             DATA_DIR / "jobwebkenya_jobs_full.csv",
    "Synthetic 40k":            DATA_DIR / "synthetic_40k.csv",
    "Synthetic Augmentation":   DATA_DIR / "synthetic_augmentation_postings.csv",
}

SCHEMA = [
    "title", "company", "company_profile", "location", "industry_category", "work_type",
    "salary_range", "experience_level", "min_qualification",
    "description", "requirements", "source_url", "source_platform",
    "is_fraud", "data_source",
]

DEDUP_COLUMNS = ["title", "company", "company_profile", "location", "description"]  

# Labels 
class_names = {0: "Legitimate", 1: "Scam"}

# Raw labels -> shared two-class labels. "Suspicious" is treated as "Fraudulent":
# both are Scam.
EMSCAD_LABEL_MAP = {0: 0, 1: 1}                # 0 legitimate, 1 fraudulent
SYNTHETIC_LABEL_MAP = {0: 0, 1: 1, 2: 1}       # 0 legitimate, 1 suspicious, 2 fraudulent

# Salary parsing 
USD_TO_KES = 130  
salary_pattern = re.compile(r"^(KES|USD)\s*([\d,]+)\s*/\s*(\w+)$", flags=re.IGNORECASE)

# Email validation
email_pattern = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

# Work-type spelling variants
WORK_TYPE_MAP = {
    "full time": "Full-time", "full-time": "Full-time", "full_time": "Full-time", "fulltime": "Full-time",
    "part time": "Part-time", "part-time": "Part-time", "part_time": "Part-time", "parttime": "Part-time",
}
EXPORT_FILE = "merged_job_scam_2class.csv"

In [43]:
def get_datasets():
    """Current version of every source frame.

    Read from the globals on every call so a frame that was re-assigned
    (for example df1 after dropping columns) is never a stale copy.
    """
    return {
        "Fake Job Postings":        df_fake,
        "Fuzu Kenya":               df_fuzu,
        "PigiaMe Kenya":            df_pigia,
        "Corporate Staffing Kenya": df_corp,
        "BrighterMonday Kenya":     df_bm,
        "JobWeb Kenya":             df_jobweb,
        "Synthetic 40k":            df1,
        "Synthetic Augmentation":   df2,
    }


def text_columns(frame):
    """Names of the columns that hold text (object / string dtypes)."""
    return [
        col for col in frame.columns
        if frame[col].dtype == "object" or pd.api.types.is_string_dtype(frame[col].dtype)
    ]


def trim_text_columns(frame):
    """Strip leading/trailing whitespace in every text column, in place.

    Returns the number of values per column that still have edge whitespace.
    """
    for col in text_columns(frame):
        frame[col] = frame[col].astype("string").str.strip()
    return {col: int((frame[col] != frame[col].str.strip()).sum()) for col in text_columns(frame)}


def normalize_work_type(series):
    """Collapse spelling variants of Full-time / Part-time; leave other values untouched."""
    if not (series.dtype == "object" or pd.api.types.is_string_dtype(series.dtype)):
        return series
    trimmed = series.astype("string").str.strip()
    mapped = trimmed.str.lower().map(WORK_TYPE_MAP)
    return mapped.fillna(trimmed).astype("string")


def parse_salary(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan, np.nan])
    m = salary_pattern.match(str(value).strip())
    if not m:
        return pd.Series([np.nan, np.nan, np.nan])
    currency, amount, period = m.groups()
    return pd.Series([currency.upper(), float(amount.replace(',', '')), period.lower()])

## Part 2: Load all source datasets

Every raw file used by the project is loaded here **once**. Later cells clean these objects in place or re-assign them; nothing reads a raw CSV again.

Six original datasets: EMSCAD (`fake_job_postings.csv`) and five Kenyan job boards. Two synthetic files: `synthetic_40k.csv` (`df1`, raw labels 0/1/2 = Legitimate/Suspicious/Fraudulent) and `synthetic_augmentation_postings.csv` (`df2`).
Both synthetic files are merged.

In [44]:
# Six original datasets
df_fake   = pd.read_csv(SOURCE_FILES["Fake Job Postings"])
df_fuzu   = pd.read_csv(SOURCE_FILES["Fuzu Kenya"])
df_pigia  = pd.read_csv(SOURCE_FILES["PigiaMe Kenya"])
df_corp   = pd.read_csv(SOURCE_FILES["Corporate Staffing Kenya"])
df_bm     = pd.read_csv(SOURCE_FILES["BrighterMonday Kenya"])
df_jobweb = pd.read_csv(SOURCE_FILES["JobWeb Kenya"])

# Synthetic datasets

df1 = pd.read_csv(SOURCE_FILES["Synthetic 40k"])
df2 = pd.read_csv(SOURCE_FILES["Synthetic Augmentation"])

for name, frame in get_datasets().items():
    print(f"{name}")
    print(f"   rows: {frame.shape[0]:,} | columns: {frame.shape[1]}")
    print(f"   column names: {frame.columns.tolist()}")

Fake Job Postings
   rows: 17,880 | columns: 18
   column names: ['    job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
Fuzu Kenya
   rows: 666 | columns: 7
   column names: ['title', 'company', 'location', 'salary_range', 'tags', 'description', 'source_url']
PigiaMe Kenya
   rows: 1,500 | columns: 8
   column names: ['title', 'company', 'work_type', 'location', 'salary_range', 'posted', 'description', 'source_url']
Corporate Staffing Kenya
   rows: 3,988 | columns: 7
   column names: ['title', 'industry', 'salary_range', 'location', 'deadline', 'description', 'source_url']
BrighterMonday Kenya
   rows: 1,964 | columns: 13
   column names: ['title', 'company', 'category', 'location', 'work_type', 'salary_range', 'min_qualification', 'experience_level', 'experie

## Part 3: Initial dataset inspection

The starting state of every source before anything is modified: structure, first rows, summary statistics, missing values and exact duplicates.

In [45]:
for name, frame in get_datasets().items():
    print("=" * 80)
    print(name)
    print("=" * 80)
    print("Shape:", frame.shape)
    print()
    frame.info()
    print("\nFirst rows:")
    display(frame.head())
    print("\nDescriptive statistics:")
    display(frame.describe(include="all").T)
    print("\nMissing values per column:")
    display(frame.isna().sum().to_frame("missing_count"))
    print("Fully-identical duplicate rows:", frame.duplicated().sum())
    print()

Fake Job Postings
Shape: (17880, 18)

<class 'pandas.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0       job_id           17880 non-null  int64
 1   title                17880 non-null  str  
 2   location             17534 non-null  str  
 3   department           6333 non-null   str  
 4   salary_range         2868 non-null   str  
 5   company_profile      14572 non-null  str  
 6   description          17879 non-null  str  
 7   requirements         15184 non-null  str  
 8   benefits             10668 non-null  str  
 9   telecommuting        17880 non-null  int64
 10  has_company_logo     17880 non-null  int64
 11  has_questions        17880 non-null  int64
 12  employment_type      14409 non-null  str  
 13  required_experience  10830 non-null  str  
 14  required_education   9775 non-null   str  
 15  industry             12977 non-null  str  


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0



Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
job_id,17880.0,NaN,NaN,NaN,8940.5,5161.655742,1.0,4470.75,8940.5,13410.25,17880.0
title,17880,11231,English Teacher Abroad,311,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,17534,3105,"GB, LND, London",718,NaN,NaN,NaN,NaN,NaN,NaN,NaN
department,6333,1337,Sales,551,NaN,NaN,NaN,NaN,NaN,NaN,NaN
salary_range,2868,874,0-0,142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_profile,14572,1709,We help teachers get safe &amp; secure jobs ab...,726,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,17879,14801,"Play with kids, get paid for it Love travel? J...",379,NaN,NaN,NaN,NaN,NaN,NaN,NaN
requirements,15184,11967,University degree required. TEFL / TESOL / CEL...,410,NaN,NaN,NaN,NaN,NaN,NaN,NaN
benefits,10668,6204,See job description,726,NaN,NaN,NaN,NaN,NaN,NaN,NaN
telecommuting,17880.0,NaN,NaN,NaN,0.042897,0.202631,0.0,0.0,0.0,0.0,1.0



Missing values per column:


...


Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
posting_id,5000,5000,SYN-000000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,5000,12,Customer Service Agent,464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,5000,2534,Salary is commensurate with experience and wil...,468,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company,5000,3905,Not disclosed,742,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agency,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,5000,9,Nyeri,598,NaN,NaN,NaN,NaN,NaN,NaN,NaN
salary,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employment_type,5000,3,Full-time,1704,NaN,NaN,NaN,NaN,NaN,NaN,NaN
email,4258,4258,browningjason@hamilton.com,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing values per column:


,missing_count
posting_id,0
title,0
description,0
company,0
agency,5000
location,0
salary,5000
employment_type,0
email,742
source,5000


Fully-identical duplicate rows: 0



### 3.1 Labeling


In [46]:
print("EMSCAD 'fraudulent' (0 = legitimate, 1 = fraudulent):")
print(df_fake["fraudulent"].value_counts(dropna=False).sort_index())

print("\nsynthetic_40k 'is_fraud':")
print(df1["is_fraud"].value_counts(dropna=False).sort_index())

print("\nsynthetic_augmentation 'is_fraud':")
print(df2["is_fraud"].value_counts(dropna=False).sort_index())

print("\nKenyan job boards, label column present?")
for name, frame in [("Fuzu Kenya", df_fuzu), ("PigiaMe Kenya", df_pigia),
                    ("Corporate Staffing Kenya", df_corp),
                    ("BrighterMonday Kenya", df_bm), ("JobWeb Kenya", df_jobweb)]:
    print(f"   {name}: {any(col in frame.columns for col in ('is_fraud', 'fraudulent'))}")

EMSCAD 'fraudulent' (0 = legitimate, 1 = fraudulent):
fraudulent
0    17014
1      866
Name: count, dtype: int64

synthetic_40k 'is_fraud':
is_fraud
0    22857
1    11429
2     5714
Name: count, dtype: int64

synthetic_augmentation 'is_fraud':
is_fraud
0    4258
1     535
2     207
Name: count, dtype: int64

Kenyan job boards, label column present?
   Fuzu Kenya: False
   PigiaMe Kenya: False
   Corporate Staffing Kenya: False
   BrighterMonday Kenya: False
   JobWeb Kenya: False


## Part 4: Standardize column names

The dataset is standardized because it is built from very different sources: EMSCAD, five Kenyan job boards and two synthetic files. Each has its own column names, missing fields and labelling scheme, so they can only be stacked into one table once they share a common schema.

In [47]:
for frame in get_datasets().values():
    frame.columns = (
        frame.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

for name, frame in get_datasets().items():
    print(f"\n{name}")
    print(frame.columns.tolist())


Fake Job Postings
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']

Fuzu Kenya
['title', 'company', 'location', 'salary_range', 'tags', 'description', 'source_url']

PigiaMe Kenya
['title', 'company', 'work_type', 'location', 'salary_range', 'posted', 'description', 'source_url']

Corporate Staffing Kenya
['title', 'industry', 'salary_range', 'location', 'deadline', 'description', 'source_url']

BrighterMonday Kenya
['title', 'company', 'category', 'location', 'work_type', 'salary_range', 'min_qualification', 'experience_level', 'experience_length', 'language_requirement', 'working_hours', 'description', 'source_url']

JobWeb Kenya
['title', 'company', 'location', 'state', 'job_type', 'job_category', 'closing_date', 'description', 'source_url']

Synthetic 40

## Part 5: Clean string / text fields

Leading and trailing whitespace is stripped from every text column of every source **before** any duplicate detection, so two copies of a posting that differ only by stray spaces are recognised as the same posting. The wording of titles and descriptions is not altered.

In [48]:
for name, frame in get_datasets().items():
    print(f"{name}: text columns -> {text_columns(frame)}")
    print("   remaining whitespace issues:", trim_text_columns(frame))

Fake Job Postings: text columns -> ['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'employment_type', 'required_experience', 'required_education', 'industry', 'function']


   remaining whitespace issues: {'title': 0, 'location': 0, 'department': 0, 'salary_range': 0, 'company_profile': 0, 'description': 0, 'requirements': 0, 'benefits': 0, 'employment_type': 0, 'required_experience': 0, 'required_education': 0, 'industry': 0, 'function': 0}
Fuzu Kenya: text columns -> ['title', 'company', 'location', 'tags', 'description', 'source_url']
   remaining whitespace issues: {'title': 0, 'company': 0, 'location': 0, 'tags': 0, 'description': 0, 'source_url': 0}
PigiaMe Kenya: text columns -> ['title', 'work_type', 'location', 'salary_range', 'posted', 'source_url']
   remaining whitespace issues: {'title': 0, 'work_type': 0, 'location': 0, 'salary_range': 0, 'posted': 0, 'source_url': 0}
Corporate Staffing Kenya: text columns -> ['title', 'industry', 'salary_range', 'location', 'deadline', 'description', 'source_url']
   remaining whitespace issues: {'title': 0, 'industry': 0, 'salary_range': 0, 'location': 0, 'deadline': 0, 'description': 0, 'source_url': 0}
B

## Part 6: Handle missing values

### 6.1 `synthetic_40k.csv` (`df1`)

`source_url` is null in the local synthetic file and carries no modelling information there, so the cleaning step removes fully null columns.

In [49]:
missing_df1 = df1.isna().mean().mul(100).round(2).sort_values(ascending=False)
display(missing_df1.to_frame('missing_percent'))

fully_null_cols = missing_df1[missing_df1 == 100].index.tolist()
print('Fully-null columns to drop:', fully_null_cols)

df1 = df1.drop(columns=fully_null_cols)
print('Shape after dropping fully-null columns:', df1.shape)

,missing_percent
source_url,100.0
company,0.0
location,0.0
industry_category,0.0
title,0.0
work_type,0.0
salary_range,0.0
min_qualification,0.0
experience_level,0.0
description,0.0


Fully-null columns to drop: ['source_url']
Shape after dropping fully-null columns: (40000, 12)


### 6.2 `synthetic_augmentation_postings.csv` (`df2`)

"agency", "salary", and "source" are 100% null across all 5,000 rows and so they are dropped

"email" is missing for ~15% of rows. Rather than dropping those rows, a "has_email" flag is kept and the NaNs are filled with an explicit placeholder instead of silently left blank.

In [50]:
missing_df2 = df2.isna().mean().mul(100).round(2).sort_values(ascending=False)
display(missing_df2.to_frame('missing_percent'))

fully_null_cols_df2 = missing_df2[missing_df2 == 100].index.tolist()
print('Fully-null columns to drop:', fully_null_cols_df2)
df2 = df2.drop(columns=fully_null_cols_df2)

df2['has_email'] = df2['email'].notna()
df2['email'] = df2['email'].fillna('not_provided')

print('\nShape after cleanup:', df2.shape)
print(df2['has_email'].value_counts())

,missing_percent
agency,100.00
salary,100.00
source,100.00
email,14.84
title,0.00
posting_id,0.00
location,0.00
company,0.00
description,0.00
employment_type,0.00


Fully-null columns to drop: ['agency', 'salary', 'source']

Shape after cleanup: (5000, 12)
has_email
True     4258
False     742
Name: count, dtype: int64


## Part 7: Normalize categorical values

Spelling variants of the work type (`full time`, `Full_Time`, `part-time`, ...) are collapsed to one label per category **before** the sources are merged and before any distribution is calculated. Values that are not a known Full-time / Part-time variant are left exactly as they are.

In [51]:
work_type_targets = {
    "Fake Job Postings":    (df_fake,   "employment_type"),
    "PigiaMe Kenya":        (df_pigia,  "work_type"),
    "BrighterMonday Kenya": (df_bm,     "work_type"),
    "JobWeb Kenya":         (df_jobweb, "job_type"),
    "Synthetic 40k":        (df1,       "work_type"),
    "Synthetic Augmentation": (df2,     "employment_type"),
}

for name, (frame, column) in work_type_targets.items():
    if column not in frame.columns:
        print(f"{name}: no '{column}' column, skipped")
        continue
    values_before = sorted(frame[column].dropna().unique())
    frame[column] = normalize_work_type(frame[column])
    values_after = sorted(frame[column].dropna().unique())
    print(f"{name} [{column}]")
    print("   before:", values_before)
    print("   after: ", values_after)

Fake Job Postings [employment_type]
   before: ['Contract', 'Full-time', 'Other', 'Part-time', 'Temporary']
   after:  ['Contract', 'Full-time', 'Other', 'Part-time', 'Temporary']
PigiaMe Kenya [work_type]
   before: ['Contract', 'Full time', 'Part time']
   after:  ['Contract', 'Full-time', 'Part-time']
BrighterMonday Kenya [work_type]
   before: []
   after:  []
JobWeb Kenya [job_type]
   before: ['Contract', 'Full-Time', 'Internship', 'Part-Time']
   after:  ['Contract', 'Full-time', 'Internship', 'Part-time']
Synthetic 40k [work_type]
   before: ['Contract', 'Full-time']
   after:  ['Contract', 'Full-time']
Synthetic Augmentation [employment_type]
   before: ['Contract', 'Full-time', 'Part-time']
   after:  ['Contract', 'Full-time', 'Part-time']


### 7.1 Validate the categorical fields of `synthetic_40k.csv`

Full value set for each categorical column so any typo, stray category, or unexpected value is caught before modelling.

In [52]:
categorical_cols_df1 = ['work_type', 'experience_level', 'min_qualification', 'industry_category']

for col in categorical_cols_df1:
    print(f'{col} ({df1[col].nunique()} unique values)')
    print(df1[col].value_counts(dropna=False))
    print()

work_type (2 unique values)
work_type
Full-time    30028
Contract      9972
Name: count, dtype: Int64

experience_level (4 unique values)
experience_level
No experience required    10104
3-5 years                  9981
Entry level                9977
1-3 years                  9938
Name: count, dtype: Int64

min_qualification (4 unique values)
min_qualification
KCSE certificate     10140
Diploma              10024
Certificate           9960
Bachelor's degree     9876
Name: count, dtype: Int64

industry_category (9 unique values)
industry_category
Healthcare            4499
Manufacturing         4496
Retail                4475
Telecommunications    4465
Hospitality           4464
Banking & Finance     4439
Logistics             4425
Insurance             4371
Construction          4366
Name: count, dtype: Int64



## Part 8: Convert data types and validate labels

Every label is converted to a number with `errors="coerce"`, and anything that is not an allowed value is printed rather than silently dropped. No label is invented: the Kenyan job boards have none and stay `NaN`.

### 8.1 `synthetic_40k.csv` (`df1`)

Confirm the raw label only ever takes the values `0` (legitimate), `1` (suspicious), `2` (fraudulent) before casting to a small nullable integer type. Anything outside that set gets surfaced. Suspicious and Fraudulent will both be mapped to Scam.

In [53]:
df1['is_fraud'] = pd.to_numeric(df1['is_fraud'], errors='coerce')

unexpected = df1.loc[~df1['is_fraud'].isin([0, 1, 2]), 'is_fraud']
print('Unexpected / non-numeric is_fraud values:', unexpected.unique())

df1['is_fraud'] = df1['is_fraud'].astype('Int8')
print(df1['is_fraud'].value_counts(dropna=False).sort_index())

Unexpected / non-numeric is_fraud values: []
is_fraud
0    22857
1    11429
2     5714
Name: count, dtype: Int64


### 8.2 `synthetic_augmentation_postings.csv` (`df2`)

In [54]:
df2['is_fraud'] = pd.to_numeric(df2['is_fraud'], errors='coerce')

unexpected_df2 = df2.loc[~df2['is_fraud'].isin([0, 1, 2]), 'is_fraud']
print('Unexpected / non-numeric is_fraud values:', unexpected_df2.unique())

df2['is_fraud'] = df2['is_fraud'].astype('Int8')
print(df2['is_fraud'].value_counts(dropna=False).sort_index())

Unexpected / non-numeric is_fraud values: []
is_fraud
0    4258
1     535
2     207
Name: count, dtype: Int64


### 8.3 EMSCAD (`df_fake`)

EMSCAD dataset has two labels: `0` = legitimate, `1` = fraudulent. 

In [55]:
df_fake["fraudulent"] = pd.to_numeric(df_fake["fraudulent"], errors="coerce")

unexpected_fake = df_fake.loc[~df_fake["fraudulent"].isin([0, 1]), "fraudulent"]
print("Unexpected / non-numeric fraudulent values:", unexpected_fake.unique())
print(df_fake["fraudulent"].value_counts(dropna=False).sort_index())

Unexpected / non-numeric fraudulent values: []
fraudulent
0    17014
1      866
Name: count, dtype: int64


## Part 9: Parse structured fields

### 9.1 Parse `salary_range` into numeric fields (`df1`)

For modelling, salary text is split into currency, numeric amount, and period. USD amounts are converted to Kenyan shillings using the explicit fixed rate below.

In [56]:
df1[['salary_currency', 'salary_amount', 'salary_period']] = df1['salary_range'].apply(parse_salary)

unparsed = df1['salary_range'].notna() & df1['salary_currency'].isna()
print('salary_range values that failed to parse:', unparsed.sum())
if unparsed.sum():
    display(df1.loc[unparsed, 'salary_range'].unique()[:10])

df1['salary_amount_kes'] = np.where(
    df1['salary_currency'] == 'USD',
    df1['salary_amount'] * USD_TO_KES,
    df1['salary_amount']
)

print(df1[['salary_range', 'salary_currency', 'salary_amount', 'salary_period', 'salary_amount_kes']].head())

salary_range values that failed to parse: 0
        salary_range salary_currency  salary_amount salary_period  \
0  KES 339,000/month             KES       339000.0         month   
1   KES 35,000/month             KES        35000.0         month   
2   KES 38,000/month             KES        38000.0         month   
3      USD 829/month             USD          829.0         month   
4   KES 83,000/month             KES        83000.0         month   

   salary_amount_kes  
0           339000.0  
1            35000.0  
2            38000.0  
3           107770.0  
4            83000.0  


### 9.2 Validate email format (`df2`)

Sanity-check every non-placeholder email against a basic "local@domain.tld" pattern and flag anything that doesn't match.

In [57]:
email_pattern = re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')

real_emails = df2.loc[df2['has_email'], 'email']
invalid_emails = real_emails[~real_emails.str.match(email_pattern)]
print(f'Invalid email formats: {len(invalid_emails)} out of {len(real_emails)}')
if len(invalid_emails):
    display(invalid_emails.head(10))

Invalid email formats: 0 out of 4258


## Part 10: Source-specific cleaning of the synthetic files

Duplicate handling and zero-variance checks that only apply to one source, followed by saving each cleaned synthetic file. These saved files are intermediate outputs; the merge below uses the cleaned in-memory `df1` and `df2`, never the raw CSVs.

### 10.1 `synthetic_40k.csv` (`df1`): duplicate check

In [58]:
exact_dupes = df1.duplicated().sum()
print('Fully-identical duplicate rows:', exact_dupes)

content_key = ['title', 'company', 'location', 'description']
near_dupes = df1[df1.duplicated(subset=content_key, keep=False)]
print('Rows sharing title+company+location+description (salary differs):', len(near_dupes))

conflict_check = near_dupes.groupby(content_key)['is_fraud'].nunique()
print('Groups with conflicting is_fraud labels:', (conflict_check > 1).sum())
before = len(df1)
df1 = df1.drop_duplicates(keep='first').reset_index(drop=True)
print(f'Rows removed as exact duplicates: {before - len(df1)}')

Fully-identical duplicate rows: 0
Rows sharing title+company+location+description (salary differs): 530
Groups with conflicting is_fraud labels: 0
Rows removed as exact duplicates: 0


No fully-identical rows exist. There ARE 530 rows that share the same "title" / "company" / "location" / "description" but a different "salary_range"so these are kept rather than dropped.

### 10.2 `synthetic_40k.csv` (`df1`): final check and save

In [59]:
print('Final shape:', df1.shape)
print()
print('Remaining missing values:')
display(df1.isna().sum().to_frame('missing_count'))

df1.to_csv('synthetic_40k_clean.csv', index=False)
print("\nSaved: synthetic_40k_clean.csv")
df1.duplicated().sum()

Final shape: (40000, 16)

Remaining missing values:


,missing_count
title,0
company,0
location,0
industry_category,0
work_type,0
salary_range,0
experience_level,0
min_qualification,0
description,0
requirements,0



Saved: synthetic_40k_clean.csv


np.int64(0)

### 10.3 `synthetic_augmentation_postings.csv` (`df2`): zero-variance columns

The local augmentation view includes constant `country` and `is_overseas` fields. They are retained here because their value may vary when other sources are combined.

In [60]:
for col in ['country', 'is_overseas']:
    print(f'{col}: {df2[col].nunique()} unique value(s) -> {df2[col].unique()}')

country: 1 unique value(s) -> <StringArray>
['Kenya']
Length: 1, dtype: string
is_overseas: 1 unique value(s) -> <StringArray>
['No']
Length: 1, dtype: string


### 10.4 `synthetic_augmentation_postings.csv` (`df2`): duplicate postings

Duplicate checking is performed on the available content fields and the cleaning output reports the rows removed.

In [61]:
content_key_df2 = ['title', 'company', 'location', 'description']

dupe_mask = df2.duplicated(subset=content_key_df2, keep=False)
print('Rows involved in content duplicates:', dupe_mask.sum())
display(df2.loc[dupe_mask, ['posting_id'] + content_key_df2 + ['is_fraud']].sort_values(content_key_df2))
before_df2 = len(df2)
df2 = df2.drop_duplicates(subset=content_key_df2, keep='first').reset_index(drop=True)
print(f'\nRows removed as duplicate postings: {before_df2 - len(df2)}')

Rows involved in content duplicates: 20


,posting_id,title,company,location,description,is_fraud
2055,SYN-002055,Administrative Assistant,Not disclosed,Eldoret,We are looking for a Administrative Assistant ...,1
4341,SYN-004341,Administrative Assistant,Not disclosed,Eldoret,We are looking for a Administrative Assistant ...,1
2226,SYN-002226,Administrative Assistant,Not disclosed,Kakamega,We are looking for a Administrative Assistant ...,1
2257,SYN-002257,Administrative Assistant,Not disclosed,Kakamega,We are looking for a Administrative Assistant ...,1
4663,SYN-004663,Administrative Assistant,Not disclosed,Kisumu,Salary is commensurate with experience and wil...,1
4769,SYN-004769,Administrative Assistant,Not disclosed,Kisumu,Salary is commensurate with experience and wil...,1
1931,SYN-001931,Delivery Driver,Not disclosed,Kakamega,We are looking for a Delivery Driver to join o...,1
4717,SYN-004717,Delivery Driver,Not disclosed,Kakamega,We are looking for a Delivery Driver to join o...,1
948,SYN-000948,Delivery Driver,Not disclosed,Kisumu,We are looking for a Delivery Driver to join o...,1
2449,SYN-002449,Delivery Driver,Not disclosed,Kisumu,We are looking for a Delivery Driver to join o...,1



Rows removed as duplicate postings: 10


### 10.5 `synthetic_augmentation_postings.csv` (`df2`): final check and save

In [62]:
print('Final shape:', df2.shape)
print()
print('Remaining missing values:')
display(df2.isna().sum().to_frame('missing_count'))

df2.to_csv('synthetic_augmentation_postings_clean.csv', index=False)
print("\nSaved: synthetic_augmentation_postings_clean.csv")
df2.duplicated().sum()

Final shape: (4990, 12)

Remaining missing values:


,missing_count
posting_id,0
title,0
description,0
company,0
location,0
employment_type,0
email,0
country,0
is_overseas,0
is_fraud,0



Saved: synthetic_augmentation_postings_clean.csv


np.int64(0)

## Part 11: Standardize every source into the shared schema

The sources are structurally incompatible: EMSCAD calls its label `fraudulent` and has no employer-name field (only a `company_profile` paragraph), Fuzu stores the category in `tags`, PigiaMe has no industry field at all, and the synthetic files use yet another layout. Each source is mapped into the same 15-column schema (`SCHEMA`, Part 1). Fields a source does not have are filled with `NaN` rather than guessed, so missing information stays honest instead of encoding which job board a row came from.

**`company` vs `company_profile`:** `company` holds an employer name only. EMSCAD has no such field, so its `company` is `NaN` and the profile paragraph is kept separately in `company_profile` (`NaN` for every other source) instead of being pushed into `company`.

**Order dependency:** `df_synthetic_std` and `df_augmentation_std` are built from the *cleaned* `df1` and `df2`. Labels are mapped to the shared two-class scheme here, before the merge: EMSCAD `0 -> 0`, `1 -> 1`; synthetic `0 -> 0`, `1` (Suspicious) `-> 1`, `2` (Fraudulent) `-> 1`. Suspicious and Fraudulent are both Scam.

In [63]:
# 1. Fake Job Postings (EMSCAD) 
df_fake_std = pd.DataFrame({
    'title': df_fake['title'],
    'company': np.nan,   
    'company_profile': df_fake['company_profile'], 
    'location': df_fake['location'],
    'industry_category': df_fake['industry'],
    'work_type': df_fake['employment_type'],
    'salary_range': df_fake['salary_range'],
    'experience_level': df_fake['required_experience'],
    'min_qualification': df_fake['required_education'],
    'description': df_fake['description'],
    'requirements': df_fake['requirements'],
    'source_url': np.nan,
    'source_platform': 'Fake Job Postings Dataset',
    'is_fraud': df_fake['fraudulent'].map(EMSCAD_LABEL_MAP),
    'data_source': 'original',
})

# 2. Fuzu Kenya
df_fuzu_std = pd.DataFrame({
    'title': df_fuzu['title'],
    'company': df_fuzu['company'],
    'company_profile': np.nan,
    'location': df_fuzu['location'],
    'industry_category': df_fuzu['tags'],
    'work_type': np.nan,
    'salary_range': df_fuzu['salary_range'],
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df_fuzu['description'],
    'requirements': np.nan,
    'source_url': df_fuzu['source_url'],
    'source_platform': 'Fuzu Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})

# 3. PigiaMe Kenya 
df_pigia_std = pd.DataFrame({
    'title': df_pigia['title'],
    'company': df_pigia['company'],
    'company_profile': np.nan,
    'location': df_pigia['location'],
    'industry_category': np.nan,
    'work_type': df_pigia['work_type'],
    'salary_range': df_pigia['salary_range'],
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df_pigia['description'],
    'requirements': np.nan,
    'source_url': df_pigia['source_url'],
    'source_platform': 'PigiaMe Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})

# 4. Corporate Staffing Kenya
df_corp_std = pd.DataFrame({
    'title': df_corp['title'],
    'company': np.nan,
    'company_profile': np.nan,
    'location': df_corp['location'],
    'industry_category': df_corp['industry'],
    'work_type': np.nan,
    'salary_range': df_corp['salary_range'],
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df_corp['description'],
    'requirements': np.nan,
    'source_url': df_corp['source_url'],
    'source_platform': 'Corporate Staffing Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})

In [64]:
# 5. BrighterMonday Kenya
df_bm_std = pd.DataFrame({
    'title': df_bm['title'],
    'company': df_bm['company'],
    'company_profile': np.nan,
    'location': df_bm['location'],
    'industry_category': df_bm['category'],
    'work_type': df_bm['work_type'],
    'salary_range': df_bm['salary_range'],
    'experience_level': df_bm['experience_level'],
    'min_qualification': df_bm['min_qualification'],
    'description': df_bm['description'],
    'requirements': np.nan,
    'source_url': df_bm['source_url'],
    'source_platform': 'BrighterMonday Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})

In [65]:
# 6. JobWeb Kenya
df_jobweb_std = pd.DataFrame({
    'title': df_jobweb['title'],
    'company': df_jobweb['company'],
    'company_profile': np.nan,
    'location': df_jobweb['location'],
    'industry_category': df_jobweb['job_category'],
    'work_type': df_jobweb['job_type'],
    'salary_range': np.nan,
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df_jobweb['description'],
    'requirements': np.nan,
    'source_url': df_jobweb['source_url'],
    'source_platform': 'JobWeb Kenya',
    'is_fraud': np.nan,
    'data_source': 'original',
})

### 11.1 Synthetic 40k (`df1`)

`synthetic_40k.csv` already uses the standard column names, so it is passed through with a `data_source` tag. Its raw 0/1/2 labels are mapped to the shared two-class scheme (Suspicious and Fraudulent both become Scam).

In [66]:
df_synthetic_std = pd.DataFrame({
    'title': df1['title'],
    'company': df1['company'],
    'company_profile': np.nan,
    'location': df1['location'],
    'industry_category': df1['industry_category'],
    'work_type': df1['work_type'],
    'salary_range': df1['salary_range'],
    'experience_level': df1['experience_level'],
    'min_qualification': df1['min_qualification'],
    'description': df1['description'],
    'requirements': df1['requirements'],
    'source_url': np.nan,
    'source_platform': 'Synthetic',
    'is_fraud': df1['is_fraud'].map(SYNTHETIC_LABEL_MAP).astype('Int8'),   # Suspicious and Fraudulent -> 1 (Scam)
    'data_source': 'synthetic',
})

### 11.2 Synthetic augmentation (`df2`)

`synthetic_augmentation_postings.csv` is on the same raw 0/1/2 label scheme and is mapped the same way (Suspicious and Fraudulent both become Scam). It is mapped into the shared schema from the *cleaned* `df2` and tagged `synthetic`, so the near-duplicate treats it like the other synthetic data (real postings win over synthetic copies). Its `salary` column was 100% null and was dropped in Part 6, so `salary_range` is `NaN`. `posting_id`, `email`, `country` and `is_overseas` are not part of the shared schema and stay only in `synthetic_augmentation_postings_clean.csv`.

In [67]:
df_augmentation_std = pd.DataFrame({
    'title': df2['title'],
    'company': df2['company'],
    'company_profile': np.nan,
    'location': df2['location'],
    'industry_category': np.nan,
   
    'work_type': df2['employment_type'] if 'employment_type' in df2.columns else np.nan,
    'salary_range': np.nan,
    'experience_level': np.nan,
    'min_qualification': np.nan,
    'description': df2['description'],
    'requirements': np.nan,
    'source_url': np.nan,
    'source_platform': 'Synthetic Augmentation',
    'is_fraud': df2['is_fraud'].map(SYNTHETIC_LABEL_MAP).astype('Int8'),
    'data_source': 'synthetic',
})

## Part 12: Combine datasets

The standardized sources are stacked **once** here. `final_df` is never rebuilt from the source frames after this cell.

In [68]:
standardized_frames = {
    "Fake Job Postings":        df_fake_std,
    "Fuzu Kenya":               df_fuzu_std,
    "PigiaMe Kenya":            df_pigia_std,
    "Corporate Staffing Kenya": df_corp_std,
    "BrighterMonday Kenya":     df_bm_std,
    "JobWeb Kenya":             df_jobweb_std,
    "Synthetic 40k":            df_synthetic_std,
    "Synthetic Augmentation":   df_augmentation_std,
}

print("Rows contributed by each source:")
for name, frame in standardized_frames.items():
    assert list(frame.columns) == SCHEMA, f"{name} does not match the shared schema"
    print(f"   {name}: {len(frame):,}")

final_df = pd.concat(
    list(standardized_frames.values()),
    ignore_index=True
)

print("\nRows after merge:", len(final_df))
print("Columns:", final_df.shape[1])
display(final_df.head(3))

Rows contributed by each source:
   Fake Job Postings: 17,880
   Fuzu Kenya: 666
   PigiaMe Kenya: 1,500
   Corporate Staffing Kenya: 3,988
   BrighterMonday Kenya: 1,964
   JobWeb Kenya: 5,323
   Synthetic 40k: 40,000
   Synthetic Augmentation: 4,990

Rows after merge: 76311
Columns: 15


,title,company,company_profile,location,industry_category,work_type,salary_range,experience_level,min_qualification,description,requirements,source_url,source_platform,is_fraud,data_source
0,Marketing Intern,NaN,"We're Food52, and we've created a groundbreaki...","US, NY, New York",<NA>,Other,<NA>,Internship,<NA>,"Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,Fake Job Postings Dataset,0.0,original
1,Customer Service - Cloud Video Production,NaN,"90 Seconds, the worlds Cloud Video Production ...","NZ, , Auckland",Marketing and Advertising,Full-time,<NA>,Not Applicable,<NA>,Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,NaN,Fake Job Postings Dataset,0.0,original
2,Commissioning Machinery Assistant (CMA),NaN,Valor Services provides Workforce Solutions th...,"US, IA, Wever",<NA>,<NA>,<NA>,<NA>,<NA>,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,Fake Job Postings Dataset,0.0,original


We combine labeled EMSCAD data, synthetic Kenya-market postings, and unlabeled job-board data. A real labeled holdout is carved out before modelling and reserved for final evaluation.

### 12.1 Check the `is_fraud` labels after the merge

All labels were mapped to the shared two-class scheme in Part 11 (`0` = Legitimate, `1` = Scam; Suspicious and Fraudulent both count as Scam), and unlabeled job-board rows remain `NaN`.

In [69]:
# Unify the dtype after stacking sources that used different integer / float types.
final_df['is_fraud'] = pd.to_numeric(final_df['is_fraud'], errors='coerce')

print('is_fraud value counts after the merge:')
print(final_df['is_fraud'].value_counts(dropna=False).sort_index())

is_fraud value counts after the merge:
is_fraud
0.0     44129
1.0     18741
<NA>    13441
Name: count, dtype: Int64


### 12.2 Inspect fraud labels by source

A key sanity check before modelling: confirms the Kenya job-board sources
are still unlabeled (`NaN`), EMSCAD has real 0/1 labels, and the synthetic
data carries both classes.

In [70]:
fraud_by_source = pd.crosstab(
    final_df['source_platform'],
    final_df['is_fraud'],
    dropna=False
)
display(fraud_by_source)

is_fraud,0.0,1.0,<NA>
source_platform,,,
BrighterMonday Kenya,0,0,1964
Corporate Staffing Kenya,0,0,3988
Fake Job Postings Dataset,17014,866,0
Fuzu Kenya,0,0,666
JobWeb Kenya,0,0,5323
PigiaMe Kenya,0,0,1500
Synthetic,22857,17143,0
Synthetic Augmentation,4258,732,0


## Part 13: Exact duplicate removal

Duplicates are defined by the project rule `DEDUP_COLUMNS` (title, company, location, description) and are removed **after** the merge, so duplicates across sources are caught too.

In [71]:
before = len(final_df)

final_df = final_df.drop_duplicates(
    subset=DEDUP_COLUMNS,
    keep='first'
).reset_index(drop=True)

after = len(final_df)
print('Rows before duplicate removal:', before)
print('Rows after duplicate removal:', after)
print('Duplicates removed:', before - after)

Rows before duplicate removal: 76311
Rows after duplicate removal: 75223
Duplicates removed: 1088


## Part 14: Near-duplicate (fingerprint) removal

Each posting is fingerprinted by normalizing and hashing its title, description and requirements text, and near-duplicates are removed. This runs **after** the merge and after exact-duplicate removal, and the result replaces `final_df`, so every later step, and the export, uses the near-deduplicated data. It also prevents leakage: no two near-identical postings can end up on opposite sides of a later data split.

In [72]:
DEDUP_TEXT_COLUMNS = ["title", "description", "requirements"]

rows_before = len(final_df)
dedup_df = final_df.copy()


# 1. Fingerprint every posting from its normalized text

normalized_text = (
    dedup_df["title"].fillna("").astype(str)
    + " "
    + dedup_df["description"].fillna("").astype(str)
    + " "
    + dedup_df["requirements"].fillna("").astype(str)
).str.lower()

normalized_text = (
    normalized_text
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

dedup_df["fingerprint"] = normalized_text.map(
    lambda text: hashlib.md5(text.encode("utf-8")).hexdigest() if text else pd.NA
)


# 2. Describe each row: synthetic or real, labelled or unlabelled

is_synthetic = (
    dedup_df["data_source"].fillna("").astype(str).str.lower().str.contains("synthetic")
    | dedup_df["source_platform"].fillna("").astype(str).str.lower().str.contains("synthetic")
)
is_labelled = dedup_df["is_fraud"].notna()
# Unlabelled rows are <NA> here; they are excluded from every label-based step
# below by is_labelled, so they are set to 0 to allow the integer cast.
fraud_risk = (dedup_df["is_fraud"] > 0).fillna(False).astype(int)


# 3. Remove duplicate families whose real labelled copies disagree

real_labelled = dedup_df["fingerprint"].notna() & ~is_synthetic & is_labelled

labels_per_fingerprint = (
    dedup_df.loc[real_labelled, ["fingerprint"]]
    .assign(fraud_risk=fraud_risk[real_labelled])
    .groupby("fingerprint")["fraud_risk"]
    .nunique()
)

conflicting_fingerprints = set(labels_per_fingerprint[labels_per_fingerprint > 1].index)
in_conflicting_family = dedup_df["fingerprint"].isin(conflicting_fingerprints)

print("Conflicting-label duplicate families removed:", len(conflicting_fingerprints))
print("Rows removed with those families:", int(in_conflicting_family.sum()))

deduped = dedup_df[~in_conflicting_family].copy()


# 4. Keep one row per fingerprint: real before synthetic, labelled before unlabelled

deduped["_priority"] = (
    is_synthetic.loc[deduped.index].astype(int) * 2
    + (~is_labelled.loc[deduped.index]).astype(int)
)

with_text = (
    deduped[deduped["fingerprint"].notna()]
    .sort_values("_priority", kind="mergesort")
    .drop_duplicates(subset="fingerprint", keep="first")
)
without_text = deduped[deduped["fingerprint"].isna()]

assert not with_text["fingerprint"].duplicated().any(), "Duplicate fingerprints remain"


# 5. Keep the surviving rows of final_df in their original order.

surviving_index = with_text.index.union(without_text.index)   # sorted union
final_df = dedup_df.loc[surviving_index].reset_index(drop=True)

near_duplicates_removed = (
    rows_before - int(in_conflicting_family.sum()) - len(final_df)
)

print("\nRows before near-duplicate removal:", rows_before)
print("Near-duplicate rows removed:", near_duplicates_removed)
print("Rows after near-duplicate removal:", len(final_df))
print("\nRows per source after de-duplication:")
display(final_df["source_platform"].value_counts())

Conflicting-label duplicate families removed: 0
Rows removed with those families: 0

Rows before near-duplicate removal: 75223
Near-duplicate rows removed: 3941
Rows after near-duplicate removal: 71282

Rows per source after de-duplication:


source_platform
Synthetic                    39432
Fake Job Postings Dataset    15537
JobWeb Kenya                  5249
Corporate Staffing Kenya      3982
Synthetic Augmentation        3256
BrighterMonday Kenya          1930
PigiaMe Kenya                 1230
Fuzu Kenya                     666
Name: count, dtype: int64

## Part 15: Final data-quality checks

Summary of the final merged dataset, still including the metadata columns, followed by the duplicate, missing-value, label and work-type checks.

In [73]:
print('Shape:', final_df.shape)

print('\nSource platform distribution:')
display(final_df['source_platform'].value_counts())

print('\nData source distribution:')
display(final_df['data_source'].value_counts())

print('\nFraud label distribution:')
display(final_df['is_fraud'].value_counts(dropna=False).sort_index())

print('\nNamed classes:')
for label, name in class_names.items():
    count = (final_df['is_fraud'] == label).sum()
    print(f'{label}: {name} -> {count:,}')
print(f"NaN / unknown labels -> {final_df['is_fraud'].isna().sum():,}")

Shape: (71282, 16)

Source platform distribution:


source_platform
Synthetic                    39432
Fake Job Postings Dataset    15537
JobWeb Kenya                  5249
Corporate Staffing Kenya      3982
Synthetic Augmentation        3256
BrighterMonday Kenya          1930
PigiaMe Kenya                 1230
Fuzu Kenya                     666
Name: count, dtype: int64


Data source distribution:


data_source
synthetic    42688
original     28594
Name: count, dtype: int64


Fraud label distribution:


is_fraud
0.0     40097
1.0     18128
<NA>    13057
Name: count, dtype: Int64


Named classes:
0: Legitimate -> 40,097
1: Scam -> 18,128
NaN / unknown labels -> 13,057


In [74]:
print("Final rows:", len(final_df))
print("Remaining exact duplicates (project key):", final_df.duplicated(subset=DEDUP_COLUMNS).sum())
print("Remaining fully-identical rows:", final_df.duplicated().sum())

print("\nMissing values per column:")
display(final_df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nLabel distribution (share of rows):")
display(final_df["is_fraud"].value_counts(normalize=True, dropna=False).sort_index().round(4))

print("\nWork-type distribution:")
display(final_df["work_type"].value_counts(dropna=False))

print("\nCompany field by source (share filled, median length in characters):")
display(
    final_df.groupby("source_platform")["company"]
    .agg(share_filled=lambda s: round(s.notna().mean(), 3),
         median_length=lambda s: s.dropna().astype(str).str.len().median())
)

Final rows: 71282
Remaining exact duplicates (project key): 0
Remaining fully-identical rows: 0

Missing values per column:


,missing_count
company_profile,58768
source_url,58225
salary_range,23687
min_qualification,21817
company,21090
experience_level,20324
requirements,18479
is_fraud,13057
work_type,9817
industry_category,8860



Label distribution (share of rows):


is_fraud
0.0     0.5625
1.0     0.2543
<NA>    0.1832
Name: proportion, dtype: Float64


Work-type distribution:


work_type
Full-time     47125
Contract      11853
NaN            6578
<NA>           3239
Part-time      1826
Temporary       231
Internship      218
Other           212
Name: count, dtype: int64


Company field by source (share filled, median length in characters):


,share_filled,median_length
source_platform,,
BrighterMonday Kenya,1.000,26.0
Corporate Staffing Kenya,0.000,NaN
Fake Job Postings Dataset,0.000,NaN
Fuzu Kenya,0.551,20.0
JobWeb Kenya,0.992,18.0
PigiaMe Kenya,0.000,NaN
Synthetic,1.000,21.0
Synthetic Augmentation,1.000,13.0


## Part 16: Leakage and feature preparation

This notebook does not split the data or build model features: train / validation / holdout partitioning, text preprocessing and feature engineering belong to the next stage. Splitting has to use the finished, near-deduplicated dataset exported below. Fingerprint uniqueness of `final_df` is verified in Part 19, so no two near-identical postings can fall on opposite sides of a split.

## Part 17: EDA-ready dataset

The metadata columns are dropped after source-level inspection. Source provenance remains available in `final_df` before this step.

`eda_df` is the single source of truth for everything after this point.

In [75]:
required_columns = [
    'title', 'company', 'company_profile', 'location', 'industry_category', 'work_type',
    'salary_range', 'experience_level', 'min_qualification',
    'description', 'requirements', 'is_fraud'
]

eda_df = final_df[required_columns].copy()

print('EDA-ready dataset shape:', eda_df.shape)
print('\nFinal columns:')
print(eda_df.columns.tolist())

for col in ['source_url', 'source_platform', 'data_source']:
    assert col not in eda_df.columns, f'{col} was not removed.'
print('\nMetadata-column check passed.')

EDA-ready dataset shape: (71282, 12)

Final columns:
['title', 'company', 'company_profile', 'location', 'industry_category', 'work_type', 'salary_range', 'experience_level', 'min_qualification', 'description', 'requirements', 'is_fraud']

Metadata-column check passed.


### 17.1 Missing-value overview (modelling columns)

In [76]:
missing_summary = (
    eda_df.isna()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(2)
    .to_frame('missing_percent')
)
display(missing_summary)

,missing_percent
company_profile,82.44
salary_range,33.23
min_qualification,30.61
company,29.59
experience_level,28.51
requirements,25.92
is_fraud,18.32
work_type,13.77
industry_category,12.43
location,6.36


### 17.2 Preview the final dataset

In [77]:
eda_df.info()
print()
display(eda_df.head())
display(eda_df.sample(min(10, len(eda_df)), random_state=42))

<class 'pandas.DataFrame'>
RangeIndex: 71282 entries, 0 to 71281
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              71281 non-null  string 
 1   company            50192 non-null  object 
 2   company_profile    12514 non-null  object 
 3   location           66750 non-null  string 
 4   industry_category  62422 non-null  object 
 5   work_type          61465 non-null  object 
 6   salary_range       47595 non-null  object 
 7   experience_level   50958 non-null  object 
 8   min_qualification  49465 non-null  object 
 9   description        70049 non-null  object 
 10  requirements       52803 non-null  object 
 11  is_fraud           58225 non-null  Float64
dtypes: Float64(1), object(9), string(2)
memory usage: 6.6+ MB



,title,company,company_profile,location,industry_category,work_type,salary_range,experience_level,min_qualification,description,requirements,is_fraud
0,Marketing Intern,NaN,"We're Food52, and we've created a groundbreaki...","US, NY, New York",<NA>,Other,<NA>,Internship,<NA>,"Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,0.0
1,Customer Service - Cloud Video Production,NaN,"90 Seconds, the worlds Cloud Video Production ...","NZ, , Auckland",Marketing and Advertising,Full-time,<NA>,Not Applicable,<NA>,Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,0.0
2,Commissioning Machinery Assistant (CMA),NaN,Valor Services provides Workforce Solutions th...,"US, IA, Wever",<NA>,<NA>,<NA>,<NA>,<NA>,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,0.0
3,Account Executive - Washington DC,NaN,Our passion for improving quality of life thro...,"US, DC, Washington",Computer Software,Full-time,<NA>,Mid-Senior level,Bachelor's Degree,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",0.0
4,Bill Review Manager,NaN,SpotSource Solutions LLC is a Global Human Cap...,"US, FL, Fort Worth",Hospital & Health Care,Full-time,<NA>,Mid-Senior level,Bachelor's Degree,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,0.0


,title,company,company_profile,location,industry_category,work_type,salary_range,experience_level,min_qualification,description,requirements,is_fraud
45428,Customer Care Agent,Rapid Placements Kenya,NaN,Nakuru,Healthcare,Full-time,"KES 312,000/month",1-3 years,Diploma,"Only 3 positions left, apply today! Customer C...","No experience needed, anyone can apply.",1.0
62434,Driver (Overseas),Global Careers Ltd,NaN,Saudi Arabia,Telecommunications,Full-time,USD 653/month,No experience required,Bachelor's degree,Global Careers Ltd is looking for a Driver (Ov...,Diploma required. No experience required prefe...,1.0
60433,Marketing Assistant,Metro HR Consultants Ltd,NaN,Nakuru,Retail,Full-time,"KES 67,000/month",3-5 years,Bachelor's degree,Metro HR Consultants Ltd is looking for a Mark...,Diploma required. Entry level preferred.,1.0
57780,Accounts Assistant,Metro Careers Ltd,NaN,Thika,Logistics,Contract,"KES 99,000/month",No experience required,Bachelor's degree,Metro Careers Ltd is looking for a Accounts As...,Bachelor's degree required. No experience requ...,1.0
48480,Procurement Officer,Java House,NaN,Nyeri,Manufacturing,Contract,"KES 79,000/month",3-5 years,KCSE certificate,Java House is seeking a qualified Procurement ...,Diploma required. Entry level preferred.,0.0
32131,Marketing Assistant,Kakuzi PLC,NaN,Machakos,Retail,Full-time,"KES 43,000/month",No experience required,Diploma,Kakuzi PLC is seeking a qualified Marketing As...,Bachelor's degree required. 1-3 years preferred.,0.0
60305,IT Support Technician,Bamburi Cement Group,NaN,Thika,Retail,Contract,"KES 68,000/month",No experience required,Certificate,Bamburi Cement Group is looking for a IT Suppo...,Certificate required. No experience required p...,1.0
56451,HR Assistant,Skyward Global Dimensions Ltd,NaN,Kericho,Hospitality,Full-time,"KES 180,000/month",Entry level,Bachelor's degree,"Hurry, recruitment closes this week! HR Assist...","No experience needed, anyone can apply.",1.0
19172,Head of Transport Department Job Mirema School...,NaN,NaN,<NA>,Logistics Salary,NaN,Open,NaN,NaN,Salary: Open\nLocation: Nairobi\nCountry: Keny...,NaN,<NA>
56364,Driver (Overseas),Ready Manpower Limited,NaN,Oman,Hospitality,Full-time,USD 791/month,Entry level,KCSE certificate,Ready Manpower Limited is seeking a qualified ...,Bachelor's degree required. 3-5 years preferred.,0.0


## Part 18: Export

Next stage: text preprocessing, feature engineering, train/test splitting and binary scam modelling.

The exact dataframe used above (`eda_df`) is exported and read back to verify the file.

In [78]:
output_file = EXPORT_FILE
eda_df.to_csv(output_file, index=False)
print(f'Saved: {output_file}')

check_df = pd.read_csv(output_file)
print("Exported rows:", len(check_df))
print("Exported columns:", len(check_df.columns))

assert len(check_df) == len(eda_df), "Exported row count does not match eda_df"
assert list(check_df.columns) == list(eda_df.columns), "Exported columns do not match eda_df"

Saved: merged_job_scam_2class.csv


C:\Users\Administrator\AppData\Local\Temp\ipykernel_5540\1980905799.py:5: DtypeWarning: Columns (0: company_profile) have mixed types. Specify dtype option on import or set low_memory=False.
  check_df = pd.read_csv(output_file)


Exported rows: 71282
Exported columns: 12


## Part 19: Final automated check

In [79]:
print("========== FINAL DATA PREPARATION CHECK ==========")
print("Final dataframe shape:", final_df.shape)
print("EDA-ready dataframe shape:", eda_df.shape)
print("Remaining exact duplicates:", final_df.duplicated().sum())
print("Remaining exact duplicates (project key):", final_df.duplicated(subset=DEDUP_COLUMNS).sum())
print("Missing is_fraud labels (unlabelled job-board rows, intentional):", final_df["is_fraud"].isna().sum())
print("Unique fingerprints:", final_df["fingerprint"].nunique())
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))
print("Fingerprint overlap between partitions: not applicable, no split in this notebook")

assert final_df.duplicated().sum() == 0
assert final_df.duplicated(subset=DEDUP_COLUMNS).sum() == 0
assert final_df["fingerprint"].dropna().is_unique, "Near-duplicate postings remain"
assert set(final_df["is_fraud"].dropna().unique()) <= {0, 1}, "Unexpected is_fraud values"
assert len(eda_df) == len(final_df)
# `company` holds employer names only: EMSCAD has none, so it must be empty for EMSCAD.
assert final_df.loc[final_df["source_platform"] == "Fake Job Postings Dataset", "company"].isna().all(), \
    "EMSCAD company_profile leaked into company"
# is_fraud is intentionally NaN for the unlabelled Kenyan job-board postings, so
# there is deliberately no assertion that every label is present.
print("\nAll checks passed.")

========== FINAL DATA PREPARATION CHECK ==========
Final dataframe shape: (71282, 16)
EDA-ready dataframe shape: (71282, 12)
Remaining exact duplicates: 0
Remaining exact duplicates (project key): 0
Missing is_fraud labels (unlabelled job-board rows, intentional): 13057
Unique fingerprints: 71280
Rows: 71282
Columns: 16
Fingerprint overlap between partitions: not applicable, no split in this notebook

All checks passed.
